# SPATIAL INTELLIGENCE

In [1]:
# This cell is not needed if you have pip installed topologicpy
#import sys
#sys.path.append("your path/topologicpy/src")

## 1. Import the needed libraries

In [2]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Grid import Grid
from topologicpy.Graph import Graph
from topologicpy.Color import Color

## 2. Check the TopologicPy Version

In [3]:
print("This tutorial requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

This tutorial requires topologicpy version 0.9.18 or newer.
The version that you are using (0.9.43) is EQUAL TO the latest version available on PyPI.


## 3. Set your renderer:
* Visual studio code: "vscode"
* Google Colab: "colab"
* Browser: "browser"

In [4]:
renderer = "vscode"

## 4. Utility functions to reset the face dictionaries and transfer dictionaries by key

In [5]:
import plotly.io as _pio

def reset_dictionaries(shell):
    faces = Topology.Faces(shell)
    for i, f in enumerate(faces):
        d = Topology.Dictionary(f)
        keys = Dictionary.Keys(d)
        for key in keys:
            if not key == "face_id":
                d = Dictionary.RemoveKey(d, key)
        f = Topology.SetDictionary(f, d)

def transfer_dicts_by_key(topologies, selectors, key):
    dicts = {}
    for t in topologies:
        d = Topology.Dictionary(t)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            dicts[str(value)] = t
    
    for s in selectors:
        d = Topology.Dictionary(s)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            f = dicts.get(str(value), None)
            if f:
                f = Topology.SetDictionary(f, d)

def show_save(*args, save_path=None, **kwargs):
    """Show topology via Topology.Show and save the figure to PNG."""
    captured = []
    orig = _pio.show
    _pio.show = lambda f, *a, **k: (captured.append(f), orig(f, *a, **k))
    Topology.Show(*args, **kwargs)
    _pio.show = orig
    if save_path and captured:
        captured[0].write_image(save_path)

## 5. Import the gallery floor plan

In [6]:
# Convert OBJ → clean Face → BREP (run once, then use ByBREPPath below)
obj_path  = r"D:\Marina\MaCAD 2025\3 SEMESTER\Graph ML\graph_ml\graph_ml\A2_Spacial_Intelligence\3D files\british_museum.obj"
brep_path = obj_path.replace(".obj", ".brep")

# 1. Load OBJ (returns a list of triangulated topologies)
topologies = Topology.ByOBJPath(obj_path)
mesh = Cluster.ByTopologies(topologies) if len(topologies) > 1 else topologies[0]

# 2. Reconstruct a clean planar Face from the triangulated mesh boundaries
triangles = Cluster.Faces(mesh)
shell      = Shell.ByFaces(triangles)
eb         = Shell.ExternalBoundary(shell)
ib_list    = Shell.InternalBoundaries(shell)
gallery    = Face.ByWires(eb, ib_list)
gallery    = Topology.RemoveCollinearEdges(gallery)
print("gallery type:", type(gallery).__name__)

# 3. Export clean BREP (overwrite any previous version)
status = Topology.ExportToBREP(gallery, path=brep_path, overwrite=True)
print("Exported BREP:", status, "->", brep_path)

gallery type: Face
Exported BREP: True -> D:\Marina\MaCAD 2025\3 SEMESTER\Graph ML\graph_ml\graph_ml\A2_Spacial_Intelligence\3D files\british_museum.brep


In [7]:
gallery = Topology.ByBREPPath(brep_path)

## 6. Show the geometry

In [8]:
show_save(gallery,
          camera=[0,0,6],
          faceColor=[210,210,250],
          faceOpacity=1,
          edgeColor="white",
          edgeWidth=3,
          showVertices=False,
          backgroundColor="black",
          width=800,
          height=600,
          renderer=renderer,
          save_path=r"D:\Marina\MaCAD 2025\3 SEMESTER\Graph ML\graph_ml\graph_ml\A2_Spacial_Intelligence\2D files\01_gallery_floor_plan.png")

ValueError: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido


## 7. Create a grid overlay

In [ ]:
b_r = Wire.BoundingRectangle(gallery)
d = Topology.Dictionary(b_r)
xmin = Dictionary.ValueAtKey(d, "xmin")
xmax = Dictionary.ValueAtKey(d, "xmax")
ymin = Dictionary.ValueAtKey(d, "ymin")
ymax = Dictionary.ValueAtKey(d, "ymax")
width = Dictionary.ValueAtKey(d, "width")
length = Dictionary.ValueAtKey(d, "length")
uRange = list(range(0,int(width)+3,3))
vRange = list(range(0,int(length)+3,3))

grid = Grid.EdgesByDistances(gallery, clip=True, uRange=uRange, vRange=vRange)

## 8. Show the geometry and the grid

In [ ]:
show_save(gallery, grid,
          camera=[0,0,6],
          faceColor=[210,210,250],
          faceOpacity=1,
          edgeColor="grey",
          edgeWidth=3,
          showVertices=False,
          backgroundColor="black",
          width=800,
          height=600,
          renderer=renderer,
          save_path=r"D:\Marina\MaCAD 2025\3 SEMESTER\Graph ML\graph_ml\graph_ml\A2_Spacial_Intelligence\2D files\02_grid_overlay.png")

## 9. Slice the floor plan with the grid to create a topologic shell

In [ ]:
shell = Topology.Slice(gallery, grid)
faces = Topology.Faces(shell)
# Assign a sequential unique face id to reference it later (e.g. "face_21")
for i, f in enumerate(faces):
    d = Dictionary.ByKeyValue("face_id", "face_"+str(i+1))
    f = Topology.SetDictionary(f, d)

## 10. Show the resulting shell

In [ ]:
show_save(shell,
          camera=[0,0,6],
          faceColor=[210,210,250],
          faceOpacity=0.9,
          edgeColor="black",
          edgeWidth=3,
          showVertices=False,
          backgroundColor="black",
          width=800,
          height=600,
          renderer=renderer,
          save_path=r"D:\Marina\MaCAD 2025\3 SEMESTER\Graph ML\graph_ml\graph_ml\A2_Spacial_Intelligence\2D files\03_sliced_shell.png")

## 11. Derive navigation and analysis graphs from the shell

In [ ]:
# Note: Graph nodes automatically inherit the dictionaries of the entities they 
navigation_graph = Graph.ByTopology(shell, direct=False, viaSharedTopologies=True)
analysis_graph = Graph.ByTopology(shell)

## 12. Derive and store the analysis graph vertices

In [ ]:
g_verts = Graph.Vertices(analysis_graph)

## 13. Show the analysis graph

In [ ]:
show_save(analysis_graph,
          camera=[0,0,6],
          vertexSize=4,
          vertexColor="red",
          edgeColor="lightgrey",
          backgroundColor="black",
          width=800,
          height=600,
          renderer=renderer,
          save_path=r"D:\Marina\MaCAD 2025\3 SEMESTER\Graph ML\graph_ml\graph_ml\A2_Spacial_Intelligence\2D files\04_analysis_graph.png")

## 14. Spatial Intelligence through Graph Analysis

### b1. Shortest Path - Montague Pl -> Museum Cafe (Use navigation graph)

In [ ]:
import time

#start_vertex = Vertex.ByCoordinates(xmin+2, ymax-2,0) # Upper left corner
#end_vertex = Vertex.ByCoordinates(xmax-2,ymin+2,0) # Lower right corner
start_vertex = Vertex.ByCoordinates(xmax-60, ymax-2,0) # Upper right entrance - Montague Place
end_vertex = Vertex.ByCoordinates(xmin+2,ymin+2,0) # Lower left corner
crg = Graph.CompiledRoutingGraph(navigation_graph, precomputeTurns=False)
start = time.time()
shortest_path = Graph.ShortestPath(crg, vertexA=start_vertex, vertexB=end_vertex)
end = time.time()
print("Shortest Path Duration:", round(end-start, 2), "seconds")

# Straighten the shortest path (optional)
start = time.time()
straight_path = Wire.Straighten(shortest_path, host=gallery)
end = time.time()
print("Straighten Wire Duration:", round(end-start, 2), "seconds")

print("Original Shortest Path Length:", round(Wire.Length(shortest_path), 2))
print("Straightened Shortened Path Length:", round(Wire.Length(straight_path), 2))
edges = Topology.Edges(shortest_path)
for edge in edges:
    d = Dictionary.ByKeysValues(["width", "color"], [7, "red"])
    edge = Topology.SetDictionary(edge, d)
edges = Topology.Edges(straight_path)
for edge in edges:
    d = Dictionary.ByKeysValues(["width", "color"], [7, "blue"])
    edge = Topology.SetDictionary(edge, d)

Shortest Path Duration: 1.99 seconds
Straighten Wire Duration: 12.49 seconds
Original Shortest Path Length: 298.61
Straightened Shortened Path Length: 261.58


In [ ]:
show_save(gallery, shortest_path, straight_path,
          camera=[0,0,6],
          faceColor=[210,210,250],
          faceOpacity=1,
          edgeColorKey="color",
          edgeWidthKey="width",
          backgroundColor="black",
          width=800,
          height=600,
          renderer=renderer,
          save_path=r"D:\Marina\MaCAD 2025\3 SEMESTER\Graph ML\graph_ml\graph_ml\A2_Spacial_Intelligence\2D files\05_shortest_path_montague_cafe.png")

### b2. Shortest Path - Main Entrance -> Mausoleum of Halikarnassos Room (Use navigation graph)

In [ ]:
import time

#start_vertex = Vertex.ByCoordinates(xmin+2, ymax-2,0) # Upper left corner
#end_vertex = Vertex.ByCoordinates(xmax-2,ymin+2,0) # Lower right corner
start_vertex = Vertex.ByCoordinates(xmax-60, ymin+18,0) # Main entrance - bottom - Great Russel Street
end_vertex = Vertex.ByCoordinates(xmin+50,ymax-55,0)  # Upper left gallery - room 21
crg = Graph.CompiledRoutingGraph(navigation_graph, precomputeTurns=False)
start = time.time()
shortest_path = Graph.ShortestPath(crg, vertexA=start_vertex, vertexB=end_vertex)
end = time.time()
print("Shortest Path Duration:", round(end-start, 2), "seconds")

# Straighten the shortest path (optional)
start = time.time()
straight_path = Wire.Straighten(shortest_path, host=gallery)
end = time.time()
print("Straighten Wire Duration:", round(end-start, 2), "seconds")

print("Original Shortest Path Length:", round(Wire.Length(shortest_path), 2))
print("Straightened Shortened Path Length:", round(Wire.Length(straight_path), 2))
edges = Topology.Edges(shortest_path)
for edge in edges:
    d = Dictionary.ByKeysValues(["width", "color"], [7, "red"])
    edge = Topology.SetDictionary(edge, d)
edges = Topology.Edges(straight_path)
for edge in edges:
    d = Dictionary.ByKeysValues(["width", "color"], [7, "blue"])
    edge = Topology.SetDictionary(edge, d)

Shortest Path Duration: 1.64 seconds
Straighten Wire Duration: 9.02 seconds
Original Shortest Path Length: 180.43
Straightened Shortened Path Length: 156.63


In [ ]:
show_save(gallery, shortest_path, straight_path,
          camera=[0,0,6],
          faceColor=[210,210,250],
          faceOpacity=1,
          edgeColorKey="color",
          edgeWidthKey="width",
          backgroundColor="black",
          width=800,
          height=600,
          renderer=renderer,
          save_path=r"D:\Marina\MaCAD 2025\3 SEMESTER\Graph ML\graph_ml\graph_ml\A2_Spacial_Intelligence\2D files\06_shortest_path_main_mausoleum.png")

### b3. Shortest Path - Main Entrance -> Parthenon Sculptures (Use navigation graph)

In [ ]:
import time

#start_vertex = Vertex.ByCoordinates(xmin+2, ymax-2,0) # Upper left corner
#end_vertex = Vertex.ByCoordinates(xmax-2,ymin+2,0) # Lower right corner
start_vertex = Vertex.ByCoordinates(xmax-60, ymin+18,0) # Main entrance - bottom - Great Russel Street
end_vertex = Vertex.ByCoordinates(xmin+10,ymax-75,0)  # Upper left gallery - room 21
crg = Graph.CompiledRoutingGraph(navigation_graph, precomputeTurns=False)
start = time.time()
shortest_path = Graph.ShortestPath(crg, vertexA=start_vertex, vertexB=end_vertex)
end = time.time()
print("Shortest Path Duration:", round(end-start, 2), "seconds")

# Straighten the shortest path (optional)
start = time.time()
straight_path = Wire.Straighten(shortest_path, host=gallery)
end = time.time()
print("Straighten Wire Duration:", round(end-start, 2), "seconds")

print("Original Shortest Path Length:", round(Wire.Length(shortest_path), 2))
print("Straightened Shortened Path Length:", round(Wire.Length(straight_path), 2))
edges = Topology.Edges(shortest_path)
for edge in edges:
    d = Dictionary.ByKeysValues(["width", "color"], [7, "red"])
    edge = Topology.SetDictionary(edge, d)
edges = Topology.Edges(straight_path)
for edge in edges:
    d = Dictionary.ByKeysValues(["width", "color"], [7, "blue"])
    edge = Topology.SetDictionary(edge, d)

Shortest Path Duration: 1.66 seconds
Straighten Wire Duration: 12.02 seconds
Original Shortest Path Length: 201.79
Straightened Shortened Path Length: 174.46


In [ ]:
show_save(gallery, shortest_path, straight_path,
          camera=[0,0,6],
          faceColor=[210,210,250],
          faceOpacity=1,
          edgeColorKey="color",
          edgeWidthKey="width",
          backgroundColor="black",
          width=800,
          height=600,
          renderer=renderer,
          save_path=r"D:\Marina\MaCAD 2025\3 SEMESTER\Graph ML\graph_ml\graph_ml\A2_Spacial_Intelligence\2D files\07_shortest_path_main_parthenon.png")

### b4. Shortest Path - Main Entrance -> Mexico Exhibition (Use navigation graph)

In [ ]:
import time

#start_vertex = Vertex.ByCoordinates(xmin+2, ymax-2,0) # Upper left corner
#end_vertex = Vertex.ByCoordinates(xmax-2,ymin+2,0) # Lower right corner
start_vertex = Vertex.ByCoordinates(xmax-60, ymin+18,0) # Main entrance - bottom - Great Russel Street
end_vertex = Vertex.ByCoordinates(xmin+50,ymax-75,0)  # Upper left gallery - room 21
crg = Graph.CompiledRoutingGraph(navigation_graph, precomputeTurns=False)
start = time.time()
shortest_path = Graph.ShortestPath(crg, vertexA=start_vertex, vertexB=end_vertex)
end = time.time()
print("Shortest Path Duration:", round(end-start, 2), "seconds")

if shortest_path is None:
    print("Error: No path found between the two vertices. Check that the coordinates are inside the gallery.")
else:
    # Straighten the shortest path (optional)
    start = time.time()
    straight_path = Wire.Straighten(shortest_path, host=gallery)
    end = time.time()
    print("Straighten Wire Duration:", round(end-start, 2), "seconds")

    print("Original Shortest Path Length:", round(Wire.Length(shortest_path), 2))
    if straight_path:
        print("Straightened Shortened Path Length:", round(Wire.Length(straight_path), 2))
    else:
        print("Wire.Straighten returned None - path could not be straightened within gallery bounds.")
        straight_path = shortest_path  # fall back to original path

    edges = Topology.Edges(shortest_path)
    for edge in edges:
        d = Dictionary.ByKeysValues(["width", "color"], [7, "red"])
        edge = Topology.SetDictionary(edge, d)
    edges = Topology.Edges(straight_path)
    for edge in edges:
        d = Dictionary.ByKeysValues(["width", "color"], [7, "blue"])
        edge = Topology.SetDictionary(edge, d)


Shortest Path Duration: 1.5 seconds
Straighten Wire Duration: 7.47 seconds
Original Shortest Path Length: 160.42
Straightened Shortened Path Length: 138.32


In [ ]:
show_save(gallery, shortest_path, straight_path,
          camera=[0,0,6],
          faceColor=[210,210,250],
          faceOpacity=1,
          edgeColorKey="color",
          edgeWidthKey="width",
          backgroundColor="black",
          width=800,
          height=600,
          renderer=renderer,
          save_path=r"D:\Marina\MaCAD 2025\3 SEMESTER\Graph ML\graph_ml\graph_ml\A2_Spacial_Intelligence\2D files\08_shortest_path_main_mexico.png")

### c. Closeness Centrality/Integration
* Closeness centrality is a graph metric that quantifies how close a node is to all other nodes by taking the reciprocal of the sum of its shortest path distances to every other node in the network.
* In space syntax, closeness centrality corresponds to global integration, measuring how spatially accessible or topologically shallow a space is within a configuration, thereby indicating its potential for movement flow and encounter density.

In [ ]:
centrality_list = Graph.ClosenessCentrality(analysis_graph, colorScale="thermal")

* Transfer the information from the graph back to the shell

In [ ]:
reset_dictionaries(shell)
faces = Topology.Faces(shell)
_ = transfer_dicts_by_key(faces, g_verts, "face_id")

In [ ]:
show_save(faces,
          faceColorKey="cc_color",
          faceOpacity=1,
          showEdges=False,
          showVertices=False,
          camera=[0,0,6],
          backgroundColor="black",
          width=800,
          height=600,
          renderer=renderer,
          save_path=r"D:\Marina\MaCAD 2025\3 SEMESTER\Graph ML\graph_ml\graph_ml\A2_Spacial_Intelligence\2D files\09_closeness_centrality.png")

### d. Betweenness Centrality/Choice
* Betweenness centrality measures how often a node lies on the shortest paths between other nodes.

In [ ]:
centrality_list = Graph.BetweennessCentrality(analysis_graph, normalize=True, colorScale="thermal")

* Transfer the information from the graph back to the shell

In [ ]:
reset_dictionaries(shell)
faces = Topology.Faces(shell)
_ = transfer_dicts_by_key(faces, g_verts, "face_id")

In [ ]:
show_save(faces,
          faceColorKey="bc_color",
          faceOpacity=1,
          showEdges=False,
          showVertices=False,
          camera=[0,0,6],
          backgroundColor="black",
          width=800,
          height=600,
          renderer=renderer,
          save_path=r"D:\Marina\MaCAD 2025\3 SEMESTER\Graph ML\graph_ml\graph_ml\A2_Spacial_Intelligence\2D files\10_betweenness_centrality.png")

In [ ]:
status = Topology.ExportToBREP(gallery, path=r"D:\Marina\MaCAD 2025\3 SEMESTER\Graph ML\graph_ml\graph_ml\A2_Spacial_Intelligence\3D files\british_museum.brep", overwrite=True)
print(status)

True


### e. Community Detection (About 5 minutes)

In [ ]:
community_list = Graph.CommunityPartition(analysis_graph, colorScale="thermal")

In [ ]:
reset_dictionaries(shell)
_ = transfer_dicts_by_key(faces, g_verts, "face_id")

In [ ]:
show_save(faces,
          faceColorKey="cp_color",
          faceOpacity=1,
          showEdges=False,
          showVertices=False,
          camera=[0,0,6],
          backgroundColor="black",
          width=800,
          height=600,
          renderer=renderer,
          save_path=r"D:\Marina\MaCAD 2025\3 SEMESTER\Graph ML\graph_ml\graph_ml\A2_Spacial_Intelligence\2D files\11_community_detection.png")

### f. Degree centrality

Bin By Dictionary Key
* Use the community partition number (or colour) to separate the faces of the shell into different bins or categories
* Derive the outer boundary of each face group (perimeter) and make a face out of that perimeter

In [ ]:
bins = Topology.BinByDictionaryKey(faces, key="community")
bin_dict = bins[0]
keys = list(bin_dict.keys())
face_groups = []
for key in keys:
    bin_faces = bin_dict[key]
    temp_shell = Shell.ByFaces(bin_faces)
    eb = Shell.ExternalBoundary(temp_shell)
    eb = Wire.RemoveCollinearEdges(eb)
    eb = Face.ByWire(eb)
    face_groups.append(eb)



In [ ]:
Topology.Show(face_groups,
              faceOpacity=1,
              showEdges=True,
              edgeWidth=8,
              edgeColor="grey",
              camera=[0,0,6],
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)

Create a new shell from the new faces

In [ ]:
new_shell = Shell.ByFaces(face_groups)


In [ ]:
Topology.Show(new_shell,
              faceOpacity=0.9,
              showEdges=True,
              showVertices=True,
              camera=[0,0,6],
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)

Create a new graph from the new shell

In [ ]:
new_graph = Graph.ByTopology(new_shell)
new_verts = Graph.Vertices(new_graph)
for v in new_verts:
    d = Dictionary.ByKeysValues(["color", "size"], ["red", 12])
    v = Topology.SetDictionary(v, d)

In [ ]:
Topology.Show(new_shell, new_graph,
              faceOpacity=0.9,
              showEdges=True,
              showVertices=True,
              vertexSizeKey="size",
              vertexColorKey="color",
              camera=[0,0,6],
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)

Compute degree centralities

In [ ]:
degree_centralities = Graph.DegreeCentrality(new_graph, normalize=False)

Transfer/Interpolate values from the new graph vertices to the original graph vertices

In [ ]:
for v in g_verts:
    new_v = Vertex.InterpolateValue(v, vertices=new_verts, n=3, key="degree_centrality")

Derive the colour of each vertex based on the interpolated value

In [ ]:
minValue = min(degree_centralities)
maxValue = max(degree_centralities)
for v in g_verts:
    d = Topology.Dictionary(v)
    d_c = Dictionary.ValueAtKey(d, "degree_centrality")
    color = Color.AnyToHex(Color.ByValueInRange(d_c, minValue=minValue, maxValue=maxValue, colorScale="thermal"))
    d = Dictionary.SetValueAtKey(d, "dc_color", color)
    d = Dictionary.SetValueAtKey(d, "size", 16)
    v = Topology.SetDictionary(v, d)

Transfer the information from the graph vertices to the faces of the original shell

In [ ]:
reset_dictionaries(shell)
_ = transfer_dicts_by_key(faces, g_verts, "face_id")

In [ ]:
show_save(faces,
          faceColorKey="dc_color",
          faceOpacity=1,
          showEdges=False,
          showVertices=False,
          vertexSizeKey="size",
          vertexColorKey="dc_color",
          camera=[0,0,6],
          backgroundColor="black",
          width=800,
          height=600,
          renderer=renderer,
          save_path=r"D:\Marina\MaCAD 2025\3 SEMESTER\Graph ML\graph_ml\graph_ml\A2_Spacial_Intelligence\2D files\12_degree_centrality.png")

## 15. Spatial Intelligence through Isovists and Graph Analysis

### g. Derive and store the graph vertices and isovist vertices

In [ ]:
g_verts = Graph.Vertices(analysis_graph)
iso_verts = Topology.Vertices(grid)

## Visibility Graph Analysis
### Create Isovists
* Time consuming (about 20 minutes!)
* Will print out errors. Ignore.

In [ ]:
isovists = []
for v in iso_verts:
    isovist = Face.Isovist(gallery, v)
    isovists.append(isovist)

Face.Isovist - Error: Could not convert visible polygon to a TopologicPy Face. Returning None.
Face.Isovist - Error: Could not convert visible polygon to a TopologicPy Face. Returning None.
Face.Isovist - Error: Could not convert visible polygon to a TopologicPy Face. Returning None.
Face.Isovist - Error: Could not convert visible polygon to a TopologicPy Face. Returning None.
Face.Isovist - Error: Could not convert visible polygon to a TopologicPy Face. Returning None.
Face.Isovist - Error: Could not convert visible polygon to a TopologicPy Face. Returning None.
Face.Isovist - Error: Could not convert visible polygon to a TopologicPy Face. Returning None.
Face.Isovist - Error: Could not convert visible polygon to a TopologicPy Face. Returning None.
Face.Isovist - Error: Could not convert visible polygon to a TopologicPy Face. Returning None.
Face.Isovist - Error: Could not convert visible polygon to a TopologicPy Face. Returning None.
Face.Isovist - Error: Could not convert visible po

In [ ]:
Topology.Show(gallery, isovists,
              faceColorKey="cp_color",
              faceOpacity=0.6,
              showEdges=False,
              showVertices=False,
              camera=[0,0,6],
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)

### Compute the visibility of each isovist viewpoint
* Calculate how many other points of the dense grid are within each isovist's face.

In [ ]:
new_verts = []
n_list = []
for i, iso in enumerate(isovists):
    if iso: # Skip is iso is None
        v = iso_verts[i]
        b_list = Vertex.IsInternal2D(g_verts, iso)
        b_list = [b for b in b_list if b]
        n = len(b_list)
        n_list.append(n)
        d = Dictionary.ByKeyValue("visibility", n)
        v = Topology.SetDictionary(v, d)
        new_verts.append(v)



### Transfer/Interpolate values from the new graph vertices to the original graph vertices

In [ ]:
for v in g_verts:
    new_v = Vertex.InterpolateValue(v, vertices=new_verts, n=2, key="visibility")

### Derive the colour of each vertex based on the interpolated value

In [ ]:
minValue = min(n_list)
maxValue = max(n_list)
for v in g_verts:
    d = Topology.Dictionary(v)
    vb = Dictionary.ValueAtKey(d, "visibility")
    color = Color.AnyToHex(Color.ByValueInRange(vb, minValue=minValue, maxValue=maxValue, colorScale="thermal"))
    d = Dictionary.SetValueAtKey(d, "vb_color", color)
    d = Dictionary.SetValueAtKey(d, "size", 16)
    v = Topology.SetDictionary(v, d)

### Transfer the information from the graph vertices to the faces of the original shell

In [ ]:
reset_dictionaries(shell)
_ = transfer_dicts_by_key(faces, g_verts, "face_id")

In [ ]:
show_save(faces,
          faceColorKey="vb_color",
          faceOpacity=1,
          showEdges=False,
          showVertices=False,
          camera=[0,0,6],
          backgroundColor="black",
          width=800,
          height=600,
          renderer=renderer,
          save_path=r"D:\Marina\MaCAD 2025\3 SEMESTER\Graph ML\graph_ml\graph_ml\A2_Spacial_Intelligence\2D files\13_visibility_isovist.png")